## SRP445934

**paper:** [PMID: 39378090](https://pmc.ncbi.nlm.nih.gov/articles/PMC11494332/) - The extension of mammalian pregnancy required taming inflammation: Independent evolution of extended placentation in the tammar wallaby, 2024

**date, curator:** 2026-06-10, Sara Carsanaro

**notes**

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Non-gravid endometrium,UBERON:0001295,endometrium,perfect match
2,Gravid endometrium,UBERON:0001295,endometrium,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP445934"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 22/22 [00:22<00:00,  1.02s/it]
12 samples dont have attributes, try to find them somewhere else
100%|███████████████████████████████████████████| 12/12 [00:20<00:00,  1.67s/it]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,,,,,,Non-gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_209NGE d20,SAMN43195232,,,,,,,,Early attachment d20 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_209NGE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,,,,,,Non-gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_118NGE d14,SAMN43195231,,,,,,,,Pre-attachment day 14 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_118NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,,,,,,Gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_20GE d20,SAMN35992848,,,,,,,,day 20 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_20GE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,,,,,,Non-gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_35NGE d14,SAMN35992847,,,,,,,,day 14 rpy non-gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,,,,,,Non-gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_109NGE d14,SAMN35992846,,,,,,,,day 14 rpy non-gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,,,,,,Gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_110GE d14,SAMN35992845,,,,,,,,day 14 rpy gravid 3,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_110GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,,,,,,Gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_109GE d14,SAMN35992844,,,,,,,,day 14 rpy gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
7,SRX20786630,SRP445934,Illumina NovaSeq 6000,SRS18072166,,,,,,Gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_35GE d14,SAMN35992843,,,,,,,,day 14 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
8,SRX20786629,SRP445934,Illumina NovaSeq 6000,SRS18072167,,,,,,Non-gravid endometrium,Adult,,,,F,,,9315,,,,,,Me13_169NGE d0,SAMN35992842,,,,,,,,0PP 0 non-gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_169NGE d0,,,,d0,,TRANSCRIPTOMIC,PolyA
9,SRX20786628,SRP445934,Illumina NovaSeq 6000,SRS18072168,,,,,,Non-gravid endometrium,Ad

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Gravid endometrium' 'Non-gravid endometrium']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0001295'
library.loc[:,'anatName'] = 'endometrium'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,,,,Non-gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_209NGE d20,SAMN43195232,,,,,,,,Early attachment d20 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_209NGE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,,,,Non-gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_118NGE d14,SAMN43195231,,,,,,,,Pre-attachment day 14 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_118NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,,,,Gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_20GE d20,SAMN35992848,,,,,,,,day 20 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_20GE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,,,,Non-gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_35NGE d14,SAMN35992847,,,,,,,,day 14 rpy non-gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,,,,Non-gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_109NGE d14,SAMN35992846,,,,,,,,day 14 rpy non-gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,UBERON:0001295,endometrium,,,,Gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_110GE d14,SAMN35992845,,,,,,,,day 14 rpy gravid 3,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_110GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,UBERON:0001295,endometrium,,,,Gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_109GE d14,SAMN35992844,,,,,,,,day 14 rpy gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
7,SRX20786630,SRP445934,Illumina NovaSeq 6000,SRS18072166,UBERON:0001295,endometrium,,,,Gravid endometrium,Adult,perfect match,not documented,,F,,,9315,,,,,,Me13_35GE d14,SAMN35992843,,,,,,,,day 14 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
8,SRX20786629,SRP445934,Illumina NovaSeq 6000,

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_209NGE d20,SAMN43195232,,,,,,,,Early attachment d20 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_209NGE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_118NGE d14,SAMN43195231,,,,,,,,Pre-attachment day 14 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_118NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_20GE d20,SAMN35992848,,,,,,,,day 20 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_20GE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_35NGE d14,SAMN35992847,,,,,,,,day 14 rpy non-gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_109NGE d14,SAMN35992846,,,,,,,,day 14 rpy non-gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_110GE d14,SAMN35992845,,,,,,,,day 14 rpy gravid 3,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_110GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,,,,,,Me13_109GE d14,SAMN35992844,,,,,,,,day 14 rpy gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
7,SRX20786630,SRP445934,Illumina NovaSeq 6000,SRS18072166,UBERON:0001295,endometrium,UBERON:0000113,post

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_209NGE d20,SAMN43195232,,,,,,,,Early attachment d20 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_209NGE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_118NGE d14,SAMN43195231,,,,,,,,Pre-attachment day 14 rpy non-gravid 3,,,12/06/2026,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_118NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_20GE d20,SAMN35992848,,,,,,,,day 20 rpy gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_20GE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_35NGE d14,SAMN35992847,,,,,,,,day 14 rpy non-gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109NGE d14,SAMN35992846,,,,,,,,day 14 rpy non-gravid 1,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_110GE d14,SAMN35992845,,,,,,,,day 14 rpy gravid 3,,,12/06/2026,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_110GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109GE d14,SAMN35992844,,,,,,,,day 14 rpy gravid 2,,,12/06/2026,RNA was extracted from snap frozen tissue using the

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_209NGE d20,SAMN43195232,,,,,,,,Early attachment d20 rpy non-gravid 3,,SAC,2026-06-10,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_209NGE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_118NGE d14,SAMN43195231,,,,,,,,Pre-attachment day 14 rpy non-gravid 3,,SAC,2026-06-10,Rna extracted using QIAGEN RNEasy mini kit. Sequencing libraries and sequencing performed at the Ramaciotti center for genomics,,Me13_118NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_20GE d20,SAMN35992848,,,,,,,,day 20 rpy gravid 1,,SAC,2026-06-10,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_20GE d20,,,,d20,,TRANSCRIPTOMIC,PolyA
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_35NGE d14,SAMN35992847,,,,,,,,day 14 rpy non-gravid 2,,SAC,2026-06-10,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_35NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109NGE d14,SAMN35992846,,,,,,,,day 14 rpy non-gravid 1,,SAC,2026-06-10,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_109NGE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_110GE d14,SAMN35992845,,,,,,,,day 14 rpy gravid 3,,SAC,2026-06-10,RNA was extracted from snap frozen tissue using the QIAGEN RNEasy mini kit. Llibrary prep and sequencing was performed by Ramaciotti Center for Genomics.,,Me13_110GE d14,,,,d14,,TRANSCRIPTOMIC,PolyA
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109GE d14,SAMN35992844,,,,,,,,day 14 rpy gravid 2,,SAC,2026-06-10,RNA was extracted from snap fr

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 39378090'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_209NGE d20,SAMN43195232,,,,,,,PMID: 39378090,Early attachment d20 rpy non-gravid 3,,SAC,2026-06-10
1,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_118NGE d14,SAMN43195231,,,,,,,PMID: 39378090,Pre-attachment day 14 rpy non-gravid 3,,SAC,2026-06-10
2,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_20GE d20,SAMN35992848,,,,,,,PMID: 39378090,day 20 rpy gravid 1,,SAC,2026-06-10
3,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_35NGE d14,SAMN35992847,,,,,,,PMID: 39378090,day 14 rpy non-gravid 2,,SAC,2026-06-10
4,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109NGE d14,SAMN35992846,,,,,,,PMID: 39378090,day 14 rpy non-gravid 1,,SAC,2026-06-10
5,SRX20786632,SRP445934,Illumina NovaSeq 6000,SRS18072169,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_110GE d14,SAMN35992845,,,,,,,PMID: 39378090,day 14 rpy gravid 3,,SAC,2026-06-10
6,SRX20786631,SRP445934,Illumina NovaSeq 6000,SRS18072172,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109GE d14,SAMN35992844,,,,,,,PMID: 39378090,day 14 rpy gravid 2,,SAC,2026-06-10
7,SRX20786630,SRP445934,Illumina NovaSeq 6000,SRS18072166,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_35GE d14,SAMN35992843,,,,,,,PMID: 39378090,day 14 rpy gravid 1,,SAC,2026-06-10
8,SRX20786629,SRP445934,Illumina NovaSeq 6000,SRS18072167,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_169NGE d0,SAMN35992842,,,,,,,PMID: 39378090,0PP 0 non-gravid 2,,SAC,2026-06-10
9,SRX20786628,SRP445934,Illumina NovaSeq 6000,SRS18072168,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_148NGE d0,SAMN35992841,,,,,,,PMID: 39378090,0PP 0 non-gravid 1,,SAC,2026-06-10


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445934,Tammar Wallaby Endometrial RNAseq data,This study aimed to examine the changes in endometrial gene expression through reproduction in the tammar wallaby.,SRA,,,,,,,PRJNA987836,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

22

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445934,Tammar Wallaby Endometrial RNAseq data,This study aimed to examine the changes in endometrial gene expression through reproduction in the tammar wallaby.,SRA,total,Bgee 1K,22,TruSeq Stranded mRNA,full_length,,PRJNA987836,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39378090'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11494332/'
experiment.loc[:,'DOI'] = '10.1073/pnas.2310047121'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445934,Tammar Wallaby Endometrial RNAseq data,This study aimed to examine the changes in endometrial gene expression through reproduction in the tammar wallaby.,SRA,total,Bgee 1K,22,TruSeq Stranded mRNA,full_length,,PRJNA987836,39378090,https://pmc.ncbi.nlm.nih.gov/articles/PMC11494332/,10.1073/pnas.2310047121,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
65386,SRX681735,SRP045516,Illumina HiSeq 2000,SRS685324,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,Adult,perfect match,not documented,perfect match,F,,,9315,,,polyA,,,Blood transcriptome from female tammar wallaby,SAMN02991078,,,,,,,PMID: 25539578,,,SAC,2026-06-12
65387,SRX681637,SRP045516,Illumina HiSeq 2000,SRS685298,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,Adult,perfect match,not documented,perfect match,M,,,9315,,,polyA,,,Blood transcriptome from male tammar wallaby,SAMN02991077,,,,,,,PMID: 25539578,,,SAC,2026-06-12
65388,SRX25708156,SRP445934,Illumina NovaSeq 6000,SRS22349273,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_209NGE d20,SAMN43195232,,,,,,,PMID: 39378090,Early attachment d20 rpy non-gravid 3,,SAC,2026-06-10
65389,SRX25708155,SRP445934,Illumina NovaSeq 6000,SRS22349272,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_118NGE d14,SAMN43195231,,,,,,,PMID: 39378090,Pre-attachment day 14 rpy non-gravid 3,,SAC,2026-06-10
65390,SRX20786635,SRP445934,Illumina NovaSeq 6000,SRS18072171,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_20GE d20,SAMN35992848,,,,,,,PMID: 39378090,day 20 rpy gravid 1,,SAC,2026-06-10
65391,SRX20786634,SRP445934,Illumina NovaSeq 6000,SRS18072174,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_35NGE d14,SAMN35992847,,,,,,,PMID: 39378090,day 14 rpy non-gravid 2,,SAC,2026-06-10
65392,SRX20786633,SRP445934,Illumina NovaSeq 6000,SRS18072170,UBERON:0001295,endometrium,UBERON:0000113,post-juvenile adult stage,,Non-gravid endometrium,Adult,perfect match,not documented,perfect match,F,,,9315,TruSeq Stranded mRNA,full_length,polyA,,,Me13_109NGE d14,SAMN35992846,,,,,,,PMID: 39378090,day 14 rpy non-gravid 1,,SAC,2026-06-10


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1250,DRP011939,marsupial transcriptome,In this progect we perfomed RNA sequencing of ...,SRA,total,Bgee 1K,3,TruSeq RNA Sample Preparation Kit,full_length,,PRJDB15650,39199849,https://pmc.ncbi.nlm.nih.gov/articles/PMC11350...,10.3390/ani14162316,,
1251,SRP045516,Macropus eugenii Transcriptome or Gene expression,Inactivation of a single X chromosome in femal...,SRA,total,Bgee 1K,3,,,,PRJNA258238,25539578,https://pmc.ncbi.nlm.nih.gov/articles/PMC4302592/,10.1186/s12862-014-0267-z,,
1252,SRP445934,Tammar Wallaby Endometrial RNAseq data,This study aimed to examine the changes in end...,SRA,total,Bgee 1K,22,TruSeq Stranded mRNA,full_length,,PRJNA987836,39378090,https://pmc.ncbi.nlm.nih.gov/articles/PMC11494...,10.1073/pnas.2310047121,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 864b35c] adding annotated bulk experiment SRP445934
 2 files changed, 23 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.52 KiB | 2.52 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: ========================================================================
remote: 
remote:     Hello all, Gitlab will be unavailable on Friday 12.06.2026 from
remote:     10:30 to 11:30 due to security maintenance works. Please refrain
remote:    from logging in during this time. Thank you for your understanding
remote: 
remote: ========================================================================
remote: 
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   e981072..864b35c  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push